### **ALUNO:** IGOR BATISTA PEREIRA

## **PROFESSOR:** ALEXANDRE NEVES LOUZADA

## **DISCIPLINA:** LINGUAGENS DE PROGRAMAÇÃO

# Análise do Mercado de Streaming no Brasil

## 1. Introdução
Como consumo, receita, assinaturas e avaliações se distribuem entre plataformas, categorias e gêneros ao longo do tempo? Este notebook documenta a análise exploratória que fundamenta o dashboard (`app.py`).

## 2. Base de dados
`dados/simulacao_streaming_brasil.csv`: base simulada, uma linha por combinação de data, plataforma, categoria, gênero e título. **Não há colunas de região, estado, plano ou preço**, portanto essas análises não são feitas.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="darkgrid", palette=["#2563EB","#38BDF8","#7C3AED","#22D3EE","#94A3B8"])
plt.rcParams.update({"figure.figsize":(9,4.5),"figure.facecolor":"#0F1420","axes.facecolor":"#151B29","text.color":"#F8FAFC","axes.labelcolor":"#F8FAFC","xtick.color":"#94A3B8","ytick.color":"#94A3B8","axes.titleweight":"bold"})

## 3. Leitura dos dados

In [ ]:
df = pd.read_csv("../dados/simulacao_streaming_brasil.csv", encoding="utf-8-sig")
print(df.shape); display(df.head()); display(df.dtypes.to_frame("tipo"))
display(df.isna().sum().to_frame("nulos").T); print("Duplicatas:", df.duplicated().sum())
display(df.describe().T)

## 4. Limpeza e preparação
Padronização de textos, conversão de tipos e datas, remoção de duplicatas/nulos e validação (valores não negativos; avaliação entre 0 e 5). Mesmas regras de `clean_data()` no app.

In [ ]:
df.columns = df.columns.str.strip().str.lower()
cat = ["plataforma","categoria","genero","titulo","horario_pico"]
for c in cat: df[c] = df[c].astype(str).str.strip().str.replace(r"\s+"," ",regex=True)
num = ["reproducoes","tempo_medio_consumo","usuarios_ativos","assinaturas","avaliacao_media","receita_plataforma"]
df[num] = df[num].apply(pd.to_numeric, errors="coerce")
df["data"] = pd.to_datetime(df["data"], errors="coerce")
n0 = len(df)
df = df.drop_duplicates().dropna(subset=["data"]+num)
df = df[(df[num]>=0).all(axis=1) & df["avaliacao_media"].between(0,5)].reset_index(drop=True)
print(f"Registros: {n0} -> {len(df)}")
for c in cat: print(c, sorted(df[c].unique()))

## 5. Engenharia de atributos

In [ ]:
df["ano"], df["mes"] = df["data"].dt.year, df["data"].dt.month
df["periodo"] = df["data"].dt.to_period("M").dt.to_timestamp()
df["receita_por_assinatura"] = df["receita_plataforma"] / df["assinaturas"].replace(0, np.nan)
df["minutos_totais_estimados"] = df["reproducoes"] * df["tempo_medio_consumo"]  # assume tempo em minutos
df[["receita_por_assinatura","minutos_totais_estimados"]].describe()

## 6. Análise exploratória
Observação: `titulo` tem poucos valores distintos e `categoria` não é coerente com a plataforma (ex.: Música em Disney+), o que é esperado em dados simulados.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4))
sns.histplot(df["tempo_medio_consumo"], bins=30, ax=ax[0]); ax[0].set_title("Tempo médio de consumo (min)")
sns.histplot(df["avaliacao_media"], bins=30, ax=ax[1]); ax[1].set_title("Avaliação média"); plt.tight_layout(); plt.show()

## 7. KPIs

In [ ]:
kpis = {"Registros":len(df),"Reproduções":df.reproducoes.sum(),"Usuários ativos (média)":df.usuarios_ativos.mean(),
"Assinaturas (média)":df.assinaturas.mean(),"Receita total (R$)":df.receita_plataforma.sum(),
"Consumo médio (min)":df.tempo_medio_consumo.mean(),"Avaliação média":df.avaliacao_media.mean()}
pd.Series(kpis).map("{:,.2f}".format).to_frame("valor")

## 8. Gráficos

In [ ]:
p = df.groupby("plataforma").agg(receita=("receita_plataforma","sum"),reproducoes=("reproducoes","sum"),tempo=("tempo_medio_consumo","mean"),avaliacao=("avaliacao_media","mean")).sort_values("receita",ascending=False)
display(p.round(2))
fig, ax = plt.subplots(1,2,figsize=(12,4))
sns.barplot(x=p.index, y=p.receita, ax=ax[0]); ax[0].set_title("Receita por plataforma"); ax[0].tick_params(axis="x", rotation=30)
sns.boxplot(data=df, x="plataforma", y="tempo_medio_consumo", ax=ax[1]); ax[1].set_title("Tempo médio por plataforma"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
anual = df.groupby(["ano","plataforma"])["receita_plataforma"].sum().reset_index()
sns.lineplot(data=anual, x="ano", y="receita_plataforma", hue="plataforma", marker="o"); plt.title("Receita anual por plataforma"); plt.show()
sns.barplot(data=df.groupby("horario_pico").reproducoes.sum().reset_index(), x="horario_pico", y="reproducoes"); plt.title("Reproduções por horário de pico"); plt.show()
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0); plt.title("Correlação de Pearson"); plt.show()

## 9. Interpretação dos resultados

In [ ]:
rec = df.groupby("plataforma").receita_plataforma.sum()
cm = df[num].corr(); par = cm.where(np.triu(np.ones(cm.shape,bool),1)).stack(); k = par.abs().idxmax()
anual_t = df.groupby("ano").receita_plataforma.sum()
print(f"- Maior receita: {rec.idxmax()} ({rec.max()/rec.sum():.1%} do total).")
print(f"- Maior tempo médio de consumo: {df.groupby('plataforma').tempo_medio_consumo.mean().idxmax()}.")
print(f"- Pico de receita anual em {anual_t.idxmax()}; variação {anual_t.iloc[0]:,.0f} -> {anual_t.iloc[-1]:,.0f} ({anual_t.iloc[-1]/anual_t.iloc[0]-1:+.1%}).")
print(f"- Correlação mais forte: {k[0]} x {k[1]} (r = {par[k]:.2f}).")

## 10. Conclusão
Os resultados acima são calculados a partir dos dados e coincidem com os textos automáticos do dashboard (sem filtros). Como a base é simulada, diferenças entre plataformas tendem a ser pequenas e as correlações fracas; os achados descrevem a base, não o mercado real.

**Limitações:** sem dados geográficos, de planos ou de preços.